## Topic 4: Indexing & HNSW Explained

### Why Do We Need Indexing?

- Suppose a vector database contains **100 document embeddings**.
- When a user asks a question, the database compares the query embedding with all 100 embeddings. This is called **Brute-Force Search**.
- For 100 vectors, this is fast.
- Now imagine the database contains **100 million embeddings**.
- Comparing the query with every vector becomes very slow.
- To solve this problem, the vector database builds an **index**.

**Analogy**

- Think of the index as a **book with chapters**.
- There is **one book (one index)** for the entire collection.
- Similar documents are grouped into the same chapter.

For example, the collection contains:

- FD Senior Citizen
- FD Bajaj Employee
- FD Cancellation Charges
- Home Loan EMI
- Credit Card Rewards

Conceptually, the index looks like:

```text
Index

Chapter 1
  FD Senior Citizen
  FD Bajaj Employee
  FD Cancellation Charges

Chapter 2
  Home Loan EMI

Chapter 3
  Credit Card Rewards
```

- When the user asks about **FD Cancellation Charges**, the database first searches the **FD chapter** and then compares only the embeddings in that chapter.
- It does not compare the query with all 100 million embeddings.
- In reality, these chapters are **not created manually**. The vector database automatically groups similar embeddings based on their mathematical similarity.

**Definition**

An **index** is a data structure that organizes similar embeddings together, allowing the database to search only the most promising vectors instead of the entire collection.

**Benefit**

- Faster similarity search.
- Scales to millions of embeddings.
- Returns highly relevant results.

---



### Exact Search vs Approximate Search

**Exact Search:** 
- Compares the query vector with **every stored vector**.
- **Advantages:** Always returns the true nearest neighbors. Highest retrieval accuracy.
- **Disadvantages:** Slow for large datasets. Search time increases as the dataset grows.

**Approximate Nearest Neighbor (ANN) Search:** 
- Searches only the **most promising candidate vectors** instead of every vector.
- **Advantages:** Extremely fast. Scales to millions of vectors. Used by most production vector databases.
- **Disadvantages:** May miss the absolute nearest vector. Returns highly relevant results instead of mathematically perfect ones.

**For RAG:** This trade-off is acceptable because retrieving a highly relevant chunk is usually sufficient.

---

### What is HNSW?

- HNSW (**Hierarchical Navigable Small World**) is one of the most popular **Approximate Nearest Neighbor (ANN)** indexing algorithms.
- Qdrant uses **HNSW** by default.
- Instead of storing vectors as a list, HNSW organizes them as a **graph** where similar vectors are connected.
- During search, it follows these connections to quickly reach vectors close to the query instead of checking every vector.

---

### How HNSW Works

- **Step 1:** Every document embedding becomes a **graph node**.
- **Step 2:** Each node is connected to several nearby nodes when the index is built.
- **Step 3:** HNSW starts from one node and follows increasingly similar neighboring nodes instead of comparing every vector.
- **Step 4:** After exploring the most promising region, it returns the **Top-K nearest vectors** with far fewer comparisons than brute-force search.

---

### Example

- **Without HNSW:** Compare the query against all **1 million vectors**.
- **With HNSW:** Navigate through connected vectors, search only a small part of the graph, and return the nearest matching documents.
- The user gets almost the same results, but much faster.

---

### Example

- Suppose an Email FAQ system stores **1 million document embeddings**.
- **Without HNSW:** Compare the query against all **1 million vectors**.
- **With HNSW:** Navigate through connected vectors, search only a small part of the graph, and return the nearest matching documents.
- The user gets almost the same results, but much faster.

---

### Why Does Qdrant Use HNSW?

- Provides an excellent balance between **speed** and **retrieval quality**.
- Offers:
  - Very fast search.
  - High retrieval accuracy.
  - Good scalability.
  - Efficient memory usage.
  - Excellent production performance.
- For most RAG applications, HNSW delivers **near-exact results** while dramatically reducing search time.

---

### Advantages

- Extremely fast similarity search.
- Scales to millions of vectors.
- High retrieval quality.
- Production-proven.
- Automatically managed by Qdrant.
- No manual graph management required.

---

### Disadvantages

- Approximate rather than perfectly exact.
- Index building takes additional time.
- Uses more memory than storing vectors alone.
- Performance depends on index configuration.

---

### Lead-Level Interview Questions

### Q1. Why do vector databases build indexes?

**Answer**

Without an index, every search compares the query with every stored vector. As datasets grow, this becomes too slow. An index reduces the number of comparisons, allowing fast similarity search even for millions of vectors.

---

### Q2. Why is Approximate Nearest Neighbor search acceptable for RAG?

**Answer**

RAG does not require the mathematically closest document. It only needs highly relevant documents that help the LLM generate a correct answer. The small loss in accuracy is outweighed by the significant improvement in search speed.

---

### Q3. Why does Qdrant use HNSW?

**Answer**

HNSW provides an excellent balance between retrieval quality, speed, memory usage, and scalability. It has become the default indexing algorithm for many production vector databases.

---

### Q4. When might brute-force search still be preferable?

**Answer**

For very small datasets where search is already fast, brute-force search is simpler and guarantees exact results. The overhead of building an index may not be worthwhile.

---

### Q5. Does HNSW improve embedding quality?

**Answer**

No.

HNSW only makes similarity search faster.

The quality of retrieval still depends primarily on the embedding model. Poor embeddings will produce poor results regardless of the indexing algorithm.

---

### Key Takeaways

- Indexing exists to avoid comparing every vector during search.
- Brute-force search is exact but does not scale well.
- HNSW is an Approximate Nearest Neighbor indexing algorithm.
- Qdrant uses HNSW automatically.
- HNSW provides fast retrieval while maintaining high accuracy.
- Better embeddings improve retrieval quality more than changing the indexing algorithm.

### Step 1: Create Qdrant Client

```python
client = QdrantClient(":memory:")
```

**What Happens?** Creates an in-memory Qdrant database. No server or Docker is required. Data exists only while the program is running.

**Theory:** `:memory:` creates a temporary Qdrant instance. Everything is stored in RAM. Data is lost when the program exits.

**Internally:** An empty database is created. No collections, vectors, or HNSW index exist yet.

---

### Step 2: Load the Embedding Model

```python
model = SentenceTransformer(MODEL_NAME)
```

**What Happens:** Loads the embedding model into memory.

**Theory:** Qdrant does not generate embeddings. Embeddings are generated outside Qdrant. The same model must be used for both documents and queries.

**Internally:** The model is ready to generate embeddings. Nothing is stored in Qdrant yet.

---

### Step 3: Create Collection

```python
create_hnsw_collection()
```

Inside:

```python
client.create_collection(...)
```

**What Happens:** Creates a collection named **fd_hnsw_demo** to store vectors, IDs, and payload. It also defines the vector size, similarity metric, and HNSW configuration.

**Theory:** A collection is similar to a SQL table. Every vector in the collection must have the same dimension (**384**) and uses **cosine similarity** for vector comparison.

**HNSW Configuration:**

```python
HnswConfigDiff(
    m=16,
    ef_construct=100
)
```

- `m` controls how many neighboring nodes each vector keeps.
- `ef_construct` controls how thoroughly Qdrant searches for good neighbors while building the graph.
- These parameters define **how the HNSW graph will be built later** during `upsert()`.

**Internally:** An empty collection is created. Qdrant stores the HNSW configuration, but no graph nodes exist yet because no vectors have been inserted.

---

### Understanding HNSW Parameters

**`m`:** Controls how many neighboring nodes each vector keeps in the graph.

- Small `m`: Lower memory usage, faster index construction, slightly lower recall.
- Large `m`: Higher memory usage, slower index construction, better recall.

**Example:** Suppose the FAQ **"How can I close my FD before maturity?"** is inserted into the graph. With `m=16`, Qdrant connects it to approximately **16 semantically similar FAQs**, such as **premature withdrawal**, **FD penalty**, and **FD closure rules**, instead of connecting it to every document.

---

**`ef_construct`:** Controls how many candidate neighbors Qdrant examines while building the graph.

- Small `ef_construct`: Faster index construction, lower-quality graph.
- Large `ef_construct`: Slower index construction, higher-quality graph, better future search accuracy.

**Example:** While inserting the same FD closure FAQ, `ef_construct=100` tells Qdrant to examine approximately **100 candidate FAQs** before selecting the best neighbors. This increases the chance of connecting the new FAQ to the most relevant FD-related documents.

---

**Important:**

- `m` decides **how many final neighbors** each vector keeps.
- `ef_construct` decides **how many candidate neighbors** Qdrant evaluates before selecting those final neighbors.
- Both parameters are used **only while building the HNSW index**.

---

### Step 4: Generate Embeddings

```python
embeddings = model.encode(text_chunks)
```

**What Happens:** Converts every document into a **384-dimensional embedding**.

**Theory:** The `all-MiniLM-L6-v2` Sentence Transformer uses a **Transformer encoder** to understand the semantic meaning of the text. It then applies **pooling** (mean pooling in this model) to combine the token embeddings into a single **384-dimensional sentence embedding**. Documents with similar meanings produce nearby vectors, while different meanings produce distant vectors.

**Example:** The FAQs **"How can I close my FD before maturity?"** and **"Can I withdraw my FD early?"** generate similar embeddings because they have similar semantic meanings, even though the wording is different.

**Internally:** The embedding model processes each document, generates token embeddings using the Transformer, applies pooling to produce one sentence embedding, and returns a 384-dimensional vector. Qdrant is not involved in this step.

---
### Step 5: Create Qdrant Points

```python
PointStruct(...)
```

**What Happens:** Creates one Qdrant Point for each document embedding.

**Theory:** A Point is the basic storage unit in Qdrant, similar to a row in a SQL table. Each Point contains:
- **ID:** Unique identifier.
- **Vector:** Embedding representing the document.
- **Payload:** Metadata used for filtering and retrieval.

**Example:** An FD FAQ can be stored as:

- **ID:** `1`
- **Vector:** 384-dimensional embedding
- **Payload:** `{"text":"How can I close my FD before maturity?","source":"fd_faq.pdf","doc_type":"faq"}`

**Internally:** Python creates Point objects in memory. These objects are not stored in Qdrant until `upsert()` is called.

---

### Step 6: Upsert Points

```python
client.upsert(...)
```

**What Happens:** Inserts all Points into Qdrant. If a Point with the same ID already exists, it is updated.

**Theory:** `upsert()` stores the ID, vector, and payload, then automatically updates the HNSW index. You never build or maintain the graph manually.

**Example:** When the FAQ **"How can I close my FD before maturity?"** is inserted:
- The embedding is stored as a vector.
- The payload (`text`, `source`, `doc_type`) is stored.
- Qdrant automatically connects the vector to nearby FD-related vectors using the HNSW settings (`m` and `ef_construct`).

**Internally:** For every Point, Qdrant:
- Stores the ID.
- Stores the vector.
- Stores the payload.
- Searches for nearby vectors.
- Creates graph connections based on `m` and `ef_construct`.
- Updates the HNSW index automatically.

This is the step where the HNSW graph is actually built.

---

### Step 7: Perform Semantic Search

```python
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=3,
)
```

**What Happens:** Converts the user query into an embedding, searches the HNSW graph, and returns the **Top-K** most similar documents.

**Theory:** The same embedding model used for documents is used to generate the query embedding. Instead of comparing the query with every stored vector, Qdrant traverses the HNSW graph to efficiently find the nearest vectors.

**Example:** If the user asks:

> **"What happens if I close my FD early?"**

Qdrant may return:

1. **Can I withdraw my FD before maturity?** *(Highest similarity because it expresses the same intent.)*
2. **What is the penalty for premature FD closure?** *(Related to the same topic but focuses on penalties.)*
3. **FD interest rate for different tenures.** *(Still related to Fixed Deposits but less relevant, so it receives a lower similarity score.)*

**Internally:** Qdrant:
- Receives the query embedding.
- Traverses the HNSW graph.
- Computes cosine similarity for the most promising candidates.
- Returns the Top-K nearest vectors with similarity scores.

---

### Step 8: Perform Filtered Search

```python
query_filter=Filter(...)
```

**What Happens:** Searches only vectors whose payload satisfies the filter condition.

**Theory:** Metadata filtering reduces the search space. Instead of searching every document, Qdrant searches only documents matching the specified payload.

**Example:** If the filter is:

```python
doc_type = "faq"
```

Only FAQ documents participate in the search. Policy documents are ignored even if they are semantically similar.

**Internally:** Qdrant applies the metadata filter while traversing the HNSW graph. Only matching vectors are considered for the final results.

---

### Step 9: Tune Search Using `hnsw_ef`

```python
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=3,
    search_params=SearchParams(
        hnsw_ef=64
    ),
)
```

**What Happens:** Controls how thoroughly Qdrant explores the HNSW graph during a search.

**Theory:** Unlike `m` and `ef_construct`, which are used while **building** the HNSW graph, `hnsw_ef` is used **every time a search is performed**. A larger value explores more graph nodes, increasing the chance of finding the true nearest neighbors, while a smaller value returns results faster.

**Search Quality**

- **Small `hnsw_ef`:** Faster search, fewer graph nodes explored, lower recall.
- **Large `hnsw_ef`:** Slower search, more graph nodes explored, higher recall.

**Example:** Suppose the user searches:

> **"What happens if I close my FD early?"**

- With a **small `hnsw_ef`**, Qdrant explores fewer graph nodes and may miss the FAQ **"Can I withdraw my FD before maturity?"**, returning another less relevant FD document instead.
- With a **large `hnsw_ef`**, Qdrant explores more of the graph and is more likely to retrieve the best matching FAQ.

**Why Doesn't the Output Change in This Demo?**

- The collection contains only **8 vectors**.
- Even a small `hnsw_ef` explores almost the entire graph.
- With millions of vectors, the difference becomes noticeable.

**Internally:** During every query, Qdrant traverses the existing HNSW graph. The `hnsw_ef` value determines how many candidate nodes are explored before returning the final Top-K results.

---

### Summary

- **`m`:** Controls how many neighboring nodes each vector keeps. Used while **building** the HNSW graph.
- **`ef_construct`:** Controls how many candidate neighbors Qdrant examines while **building** the graph.
- **`hnsw_ef`:** Controls how thoroughly Qdrant explores the graph during **search**.

In [1]:
"""
qdrant_hnsw.py
----------------
Demonstrates HNSW index parameter control in Qdrant.
Uses :memory: mode -- no Docker required.

Install: pip install qdrant-client sentence-transformers
"""

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,       # tells Qdrant how to measure closeness between vectors
    VectorParams,   # defines the vector size and distance metric for a collection
    HnswConfigDiff, # lets us override the default HNSW index parameters
    PointStruct,    # the object that holds one point: id + vector + payload
    Filter,         # wraps one or more conditions for a filtered search
    FieldCondition, # one condition: "this payload field must match this value"
    MatchValue,     # the "must equal this exact value" part of a FieldCondition
    SearchParams,   # lets us control ef (search-time recall vs. speed trade-off)
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "fd_hnsw_demo"
VECTOR_SIZE = 384       # must match the embedding model's output dimension
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

# :memory: means no Docker, no server -- everything lives in this Python process
client = QdrantClient(":memory:")

# load the embedding model once -- reused for every encode() call below
model = SentenceTransformer(MODEL_NAME)

# small set of sample chunks -- in a real pipeline these come from chunker.py
CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to the death of the depositor.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.", "source": "FD_Product_Guide", "doc_type": "product"},
    {"text": "What is the minimum amount to open an FD? The minimum deposit is Rs 25,000.", "source": "FD_FAQ", "doc_type": "faq"},
    {"text": "Can I withdraw my FD before maturity? Yes, subject to a penalty.", "source": "FD_FAQ", "doc_type": "faq"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.", "source": "FD_Product_Guide", "doc_type": "product"},
    {"text": "Nomination facility is available for all Fixed Deposit accounts.", "source": "FD_Policy", "doc_type": "policy"},
    {"text": "TDS is deducted at source if interest income exceeds Rs 40,000 per year.", "source": "FD_Policy", "doc_type": "policy"},
]


def create_hnsw_collection(m: int = 16, ef_construction: int = 100) -> None:
    # check what collections already exist so we don't crash on a duplicate name
    existing = [c.name for c in client.get_collections().collections]

    # delete the old collection if it exists -- clean slate for this demo
    if COLLECTION_NAME in existing:
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,       # every vector in this collection must be 384 floats
            distance=Distance.COSINE,  # cosine similarity -- right choice for normalized vectors
        ),
        hnsw_config=HnswConfigDiff(
            m=m,                    # edges per node -- higher = better recall, more memory
            ef_construct=ef_construction,  # build-time search depth -- higher = better index quality
        ),
    )
    print(f"Created collection with HNSW m={m}, ef_construction={ef_construction}")


def upsert_chunks() -> None:
    # extract just the text strings so we can embed them all in one batch call
    texts = [c["text"] for c in CHUNKS]

    # embed all texts at once -- normalize_embeddings=True makes dot product == cosine similarity
    embeddings = model.encode(texts, normalize_embeddings=True)

    # build a PointStruct for each chunk: id + vector + payload (metadata)
    points = [
        PointStruct(
            id=i,                           # simple integer ID -- fine for a demo
            vector=embeddings[i].tolist(),  # numpy array -> plain Python list for serialization
            payload={
                "text": CHUNKS[i]["text"],      # store original text so search results are self-contained
                "source": CHUNKS[i]["source"],  # which source file this chunk came from
                "doc_type": CHUNKS[i]["doc_type"],  # used for filtering later
            },
        )
        for i in range(len(CHUNKS))
    ]

    # upsert = insert if new, replace if ID already exists
    client.upsert(collection_name=COLLECTION_NAME, points=points)

    # confirm how many points are now in the collection
    info = client.get_collection(COLLECTION_NAME)
    print(f"Upserted {info.points_count} points")


def search_unfiltered(query: str, k: int = 3, ef: int = 128) -> None:
    # embed the query with the same model used at ingest time -- must always match
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    # query_points replaces the old client.search() which was removed in qdrant-client >= 1.9
    # .points unwraps the result object to get the actual list of scored hits
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,         # the embedded query vector to search against
        limit=k,                    # return the top-k most similar points
        search_params=SearchParams(
            hnsw_ef=ef              # higher ef = more thorough search = better recall, slower
        ),
        with_payload=True,          # include the payload (text, source, doc_type) in results
    ).points                        # .points extracts the list from the response wrapper

    print(f"\nUnfiltered search: '{query}' (ef={ef})")
    for r in results:
        # r.score is the cosine similarity -- closer to 1.0 = more similar
        print(f"  score={r.score:.4f} | {r.payload['text'][:70]}")


def search_filtered(query: str, doc_type: str, k: int = 3) -> None:
    # same embedding step as unfiltered search -- model must always be the same
    query_vector = model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=Filter(
            must=[                          # ALL conditions in must[] must be true
                FieldCondition(
                    key="doc_type",         # the payload field to filter on
                    match=MatchValue(value=doc_type),  # must equal this exact value
                )
            ]
        ),
        limit=k,
        with_payload=True,
    ).points

    print(f"\nFiltered search (doc_type='{doc_type}'): '{query}'")
    for r in results:
        print(f"  score={r.score:.4f} | [{r.payload['doc_type']}] {r.payload['text'][:60]}")


# ----------------------------------------------------------------------
# Run all demos in order
# ----------------------------------------------------------------------

# step 1: create the collection with explicit HNSW settings
create_hnsw_collection(m=16, ef_construction=100)

# step 2: embed and store all sample chunks
upsert_chunks()

# step 3: search across all doc types -- no filter
search_unfiltered("What happens if I close my FD early?", k=3, ef=64)

# step 4: same query but only look inside FAQ documents
search_filtered("What happens if I close my FD early?", doc_type="faq", k=2)

# step 5: different query, only look inside policy documents
search_filtered("penalty for early closure", doc_type="policy", k=3)

# step 6: compare low ef vs high ef on the same query
# at 8 points the scores will be identical -- the difference shows at scale
print("\n--- ef comparison (difference visible at scale, not at 8 points) ---")
search_unfiltered("senior citizen interest rate", k=2, ef=10)   # fast, lower recall at scale
search_unfiltered("senior citizen interest rate", k=2, ef=200)  # slower, higher recall at scale


Created collection with HNSW m=16, ef_construction=100
Upserted 8 points

Unfiltered search: 'What happens if I close my FD early?' (ef=64)
  score=0.5106 | Can I withdraw my FD before maturity? Yes, subject to a penalty.
  score=0.4080 | This penalty does not apply if the FD is closed due to the death of th
  score=0.3879 | The FD interest rate for 24 months is 7.1 percent per annum.

Filtered search (doc_type='faq'): 'What happens if I close my FD early?'
  score=0.5106 | [faq] Can I withdraw my FD before maturity? Yes, subject to a pena
  score=0.3675 | [faq] What is the minimum amount to open an FD? The minimum deposi

Filtered search (doc_type='policy'): 'penalty for early closure'
  score=0.6175 | [policy] Premature withdrawal incurs a 1 percent penalty on the appli
  score=0.4741 | [policy] This penalty does not apply if the FD is closed due to the d
  score=0.0928 | [policy] TDS is deducted at source if interest income exceeds Rs 40,0

--- ef comparison (difference visible at 